# Lezione 7 — Pratica: apprendimento per rinforzo quantistico

Nella lezione teorica hai addestrato un agente QDQN con una politica epsilon-greedy che combina esplorazione e sfruttamento. Ora verificherai che cosa succede se elimini completamente lo sfruttamento e lasci che l'agente esplori sempre a caso.

## Traccia

Nella funzione `elegir_accion` della classe `AgenteQDQN`, sostituisci la politica epsilon-greedy con una politica totalmente casuale: elimina la condizione `if random.random() < self.epsilon` e fai in modo che la funzione restituisca sempre `np.random.choice(self.num_acciones)`, senza consultare mai la rete quantistica. Riaddestra l'agente per 50 episodi e confronta la curva delle ricompense con quella dell'esempio originale. Spiega in una frase perché l'esplorazione pura, senza sfruttamento, non permette all'agente di migliorare nel tempo.

**Suggerimenti**

- La funzione `elegir_accion` è l'unica parte del codice che devi modificare; la classe `AgenteQDQN` completa è già scritta più in basso
- Una politica totalmente casuale non usa mai `self.red_q`, quindi il circuito quantistico addestrato smette di influenzare le decisioni dell'agente
- La memoria di replay e l'addestramento della rete continuano a funzionare come prima: cambia solo il modo in cui viene scelta ogni azione
- Confronta la lista `recompensas_por_episodio` di questo esperimento con quella dell'esempio teorico e osserva se c'è una tendenza a salire o se resta piatta

In [ ]:
import gym
import numpy as np
import random
from collections import deque
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy

class AgenteQDQN:
    def __init__(self, num_acciones, num_variables_estado):
        self.num_acciones = num_acciones
        self.qubits = [cirq.GridQubit(0, i) for i in range(4)]
        self.red_q = self._construir_red()
        self.opt = tf.keras.optimizers.Adam(learning_rate=0.01)
        self.memoria = deque(maxlen=10000)
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_decaimiento = 0.9
        self.epsilon_minimo = 0.01

    def _rotacion_de_un_qubit(self, qubit, simbolos):
        return cirq.Circuit(cirq.X(qubit) ** simbolos[0], cirq.Y(qubit) ** simbolos[1], cirq.Z(qubit) ** simbolos[2])

    def _agrupamiento(self, fuente, receptor, simbolos):
        circuito = cirq.Circuit()
        selector_receptor = self._rotacion_de_un_qubit(receptor, simbolos[0:3])
        selector_fuente = self._rotacion_de_un_qubit(fuente, simbolos[3:6])
        circuito.append(selector_receptor)
        circuito.append(selector_fuente)
        circuito.append(cirq.CNOT(control=fuente, target=receptor))
        circuito.append(selector_receptor ** -1)
        return circuito

    def _capa(self, pesos, qubits):
        capa = cirq.Circuit()
        for i in range(len(qubits) - 1):
            capa.append(cirq.CNOT(qubits[i], qubits[i + 1]))
        capa.append([cirq.Moment([cirq.rx(pesos[j]).on(qubits[j]) for j in range(4)])])
        capa.append([cirq.Moment([cirq.ry(pesos[j + 4]).on(qubits[j]) for j in range(4)])])
        capa.append([cirq.Moment([cirq.rz(pesos[j + 8]).on(qubits[j]) for j in range(4)])])
        return capa

    def _construir_circuito(self):
        simbolos = sympy.symbols("q0:48")  # 3 capas x 12 + 2 agrupamientos x 6 = 48
        circuito = cirq.Circuit()
        circuito += self._capa(simbolos[:12], self.qubits)
        circuito += self._capa(simbolos[12:24], self.qubits)
        circuito += self._capa(simbolos[24:36], self.qubits)
        circuito += self._agrupamiento(self.qubits[0], self.qubits[2], simbolos[36:42])
        circuito += self._agrupamiento(self.qubits[1], self.qubits[3], simbolos[42:])
        return circuito

    def _construir_red(self):
        operadores_lectura = [cirq.Z(self.qubits[i]) for i in range(2, 4)]  # 2 acciones posibles
        entradas = tf.keras.Input(shape=(), dtype=tf.dtypes.string)
        capa_cuantica = tfq.layers.PQC(
            self._construir_circuito(), operadores_lectura,
            differentiator=tfq.differentiators.ParameterShift(),
            initializer=tf.keras.initializers.Zeros,
        )(entradas)
        return tf.keras.Model(inputs=entradas, outputs=capa_cuantica)

    def _codificar_estado(self, estado_clasico):
        circuito = cirq.Circuit()
        for i, valor in enumerate(estado_clasico):
            angulo = 0 if valor < 0 else 1
            circuito.append(cirq.rx(np.pi * angulo).on(self.qubits[i]))
            circuito.append(cirq.rz(np.pi * angulo).on(self.qubits[i]))
        return circuito

    def elegir_accion(self, estado):
        # TODO: sustituye esta política epsilon-greedy por una política
        # totalmente aleatoria. Elimina la condición de abajo y haz que la
        # función siempre devuelva np.random.choice(self.num_acciones),
        # sin consultar nunca self.red_q
        if random.random() < self.epsilon:
            return np.random.choice(self.num_acciones)
        entrada = tfq.convert_to_tensor([self._codificar_estado(estado)])
        return np.argmax(self.red_q.predict(entrada, verbose=0))

    def recordar(self, estado, accion, recompensa, siguiente_estado, terminado):
        self.memoria.append((estado, accion, recompensa, siguiente_estado, terminado))

    def entrenar(self, tamano_lote=32):
        if len(self.memoria) < tamano_lote:
            return
        lote = random.sample(self.memoria, tamano_lote)
        estados = tfq.convert_to_tensor([self._codificar_estado(e[0]) for e in lote])
        siguientes = tfq.convert_to_tensor([self._codificar_estado(e[3]) for e in lote])
        acciones = [[i, lote[i][1]] for i in range(tamano_lote)]
        recompensas = tf.convert_to_tensor([[e[2]] for e in lote], dtype=tf.float32)
        terminados = tf.convert_to_tensor([[float(e[4])] for e in lote], dtype=tf.float32)

        with tf.GradientTape() as cinta:
            q_siguiente = tf.reduce_max(self.red_q(siguientes), axis=1, keepdims=True)
            objetivo = recompensas + (1 - terminados) * self.gamma * q_siguiente
            q_actual = tf.gather_nd(self.red_q(estados, training=True), acciones)
            error = tf.reduce_mean(tf.square(objetivo - tf.reshape(q_actual, [tamano_lote, 1])))

        gradientes = cinta.gradient(error, self.red_q.trainable_variables)
        self.opt.apply_gradients(zip(gradientes, self.red_q.trainable_variables))
        if self.epsilon > self.epsilon_minimo:
            self.epsilon *= self.epsilon_decaimiento

In [ ]:
entorno = gym.make("CartPole-v1")
agente = AgenteQDQN(entorno.action_space.n, entorno.observation_space.shape[0])
recompensas_por_episodio = []

for episodio in range(50):
    estado = entorno.reset()
    recompensa_total = 0
    terminado = False
    while not terminado:
        accion = agente.elegir_accion(estado)
        siguiente_estado, recompensa, terminado, _ = entorno.step(accion)
        recompensa_total += recompensa
        agente.recordar(estado, accion, recompensa, siguiente_estado, terminado)
        estado = siguiente_estado
        if terminado:
            agente.entrenar()
    recompensas_por_episodio.append(recompensa_total)
    print(f"Episodio {episodio}: recompensa {recompensa_total}, epsilon {agente.epsilon:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(recompensas_por_episodio)
plt.xlabel("Episodio")
plt.ylabel("Recompensa")
plt.title("Recompensa por episodio con política totalmente aleatoria")
plt.show()

<details>
<summary>💡 Mostra soluzione</summary>

```python
def elegir_accion(self, estado):
    # Soluzione: politica totalmente casuale, senza sfruttamento.
    # Non consultiamo più self.epsilon né self.red_q: l'agente sceglie
    # sempre un'azione a caso, quindi il circuito quantistico addestrato
    # non influenza mai il suo comportamento.
    return np.random.choice(self.num_acciones)

# Perché la curva delle ricompense non migliora nel tempo:
# il metodo entrenar() continua a regolare in background i pesi del circuito,
# ma elegir_accion non consulta più quel circuito per decidere. Senza una fase
# di sfruttamento che traduca ciò che è stato appreso in decisioni, la conoscenza
# accumulata nella rete Q non si riflette nel comportamento dell'agente: la
# ricompensa per episodio resta piatta e irregolare, invece di salire come accade
# con epsilon-greedy.
```

</details>